# KuaiRand Exploratory Data Analysis

KuaiRand is a recommendation-log dataset from Kuaishou. It contains standard recommendation traffic, randomized exposure traffic, sequential interaction histories, user features, item/video features, and multiple feedback signals. The release is available in three versions: KuaiRand-Pure, KuaiRand-1K, and KuaiRand-27K.

This notebook is written as a research EDA document for recommender systems work. It intentionally separates values copied from documentation from values computed from local files. It does not download data, does not create synthetic replacements, and does not invent missing results.

## Research Questions

1. How do Pure / 1K / 27K differ?
2. How are the interaction logs organized?
3. How do standard and random exposure differ?
4. How long are user histories?
5. How sparse are the observed interactions?
6. Which feedback targets are available?
7. How is the temporal structure organized?
8. Which features may create leakage?
9. Which benchmark protocol families are possible?
10. Which version should be used for development and full experiments?


## Configuration

The notebook is meant to be executed on cHARISMa after `git pull`. The default `DATA_ROOT` is the server path `/home/daryumin/iberdov/Corpora`. A different location can be provided with the environment variable `KUAIRAND_DATA_ROOT`, but no local fallback dataset is used automatically.


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "eda_utils.py").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.eda_utils import (  # noqa: E402
    FEEDBACK_COLUMNS,
    INTERACTION_COLUMNS,
    WATCH_TIME_COLUMNS,
    available_columns,
    collect_lazy,
    concat_lazy_frames,
    dataset_inventory,
    discover_kuairand_files,
    human_size,
    lazy_schema_names,
    numeric_summary,
    require_polars,
    safe_percentiles,
    scan_table,
    total_size_bytes,
    version_roots,
)

SERVER_DATA_ROOT = Path("/home/daryumin/iberdov/Corpora")
DATA_ROOT = Path(os.environ.get("KUAIRAND_DATA_ROOT", SERVER_DATA_ROOT)).expanduser()
PURE_ROOT = DATA_ROOT / "KuaiRand-Pure" / "KuaiRand-Pure"
K1_ROOT = DATA_ROOT / "KuaiRand-1K" / "KuaiRand-1K"
K27_ROOT = DATA_ROOT / "KuaiRand-27K" / "KuaiRand-27K"
VERSION_ROOTS = version_roots(DATA_ROOT)
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "eda"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)
plt.style.use("seaborn-v0_8-whitegrid")

DATA_AVAILABLE = DATA_ROOT.exists()
VERSION_AVAILABLE = {version: root.exists() for version, root in VERSION_ROOTS.items()}

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"DATA_ROOT = {DATA_ROOT}")
if not DATA_AVAILABLE:
    print(f"KuaiRand data directory not found: {DATA_ROOT}")
else:
    display(pd.DataFrame(
        [
            {"version": version, "path": str(root), "exists": exists}
            for version, root in VERSION_ROOTS.items()
            for exists in [root.exists()]
        ]
    ))


## Dataset Versions

The following table contains **Documentation / README statistics**. These numbers are copied from the dataset documentation and must not be mixed with values computed by this notebook from files available on the machine.


In [ ]:
documentation_stats = pd.DataFrame([
    {
        "version": "KuaiRand-27K",
        "users": 27_285,
        "standard_items": 32_038_725,
        "standard_interactions": 322_278_385,
        "random_items": 7_583,
        "random_interactions": 1_186_059,
        "source": "Documentation / README statistics",
    },
    {
        "version": "KuaiRand-1K",
        "users": 1_000,
        "standard_items": 4_369_953,
        "standard_interactions": 11_713_045,
        "random_items": 7_388,
        "random_interactions": 43_028,
        "source": "Documentation / README statistics",
    },
    {
        "version": "KuaiRand-Pure",
        "users": 27_285,
        "standard_items": "about 7.5K candidate-pool items",
        "standard_interactions": 1_436_609,
        "random_items": 7_583,
        "random_interactions": 1_186_059,
        "source": "Documentation / README statistics",
    },
])
display(documentation_stats)


## Dataset Inventory

This section scans directory metadata only. It lists files, file sizes, suffixes, and EDA categories for Pure / 1K / 27K when the data directories exist.


In [ ]:
inventory_rows = []
for version, root in VERSION_ROOTS.items():
    rows = dataset_inventory(root) if root.exists() else []
    for row in rows:
        inventory_rows.append({"version": version, **row})

inventory_df = pd.DataFrame(inventory_rows)
if inventory_df.empty:
    print("Skipped inventory: KuaiRand data directories are not available in DATA_ROOT.")
else:
    display(inventory_df.sort_values(["version", "category", "path"]))

    inventory_summary = (
        inventory_df.groupby(["version", "category"], as_index=False)
        .agg(files=("path", "count"), size_bytes=("size_bytes", "sum"))
        .assign(size=lambda frame: frame["size_bytes"].map(human_size))
        .sort_values(["version", "category"])
    )
    display(inventory_summary)


In [ ]:
disk_summary_rows = []
for version, root in VERSION_ROOTS.items():
    rows = dataset_inventory(root) if root.exists() else []
    disk_summary_rows.append({
        "version": version,
        "path": str(root),
        "exists": root.exists(),
        "files": len(rows),
        "disk_size_bytes": total_size_bytes(rows),
        "disk_size": human_size(total_size_bytes(rows)) if rows else None,
        "source": "Computed from local files" if rows else "Not computed: data directory missing",
    })

disk_summary = pd.DataFrame(disk_summary_rows)
display(disk_summary)


## File Discovery

The code below discovers expected KuaiRand tables by filename tokens instead of hard-coding version-specific file suffixes. This makes the notebook robust to Pure / 1K / 27K filename variants.


In [ ]:
DISCOVERED_FILES = {
    version: discover_kuairand_files(root) if root.exists() else {}
    for version, root in VERSION_ROOTS.items()
}

discovered_rows = []
for version, files in DISCOVERED_FILES.items():
    for logical_name, path in files.items():
        discovered_rows.append({
            "version": version,
            "logical_name": logical_name,
            "path": str(path) if path is not None else None,
            "exists": path is not None,
        })

discovered_files_df = pd.DataFrame(discovered_rows)
if discovered_files_df.empty:
    print("Skipped file discovery: no version directory is available.")
else:
    display(discovered_files_df.sort_values(["version", "logical_name"]))


## Standard Recommendation vs Random Exposure

KuaiRand separates three interaction-log periods:

- `log_standard_4_08_to_4_21`
- `log_standard_4_22_to_5_08`
- `log_random_4_22_to_5_08`

These names describe collection policy and calendar windows. They should not be automatically renamed to train / validation / test. A benchmark must define its own preprocessing, split, candidate set, and evaluation protocol.

The standard logs come from the platform recommendation policy. The random log records randomized exposure from a candidate pool. Relevant fields include `is_rand`, `tab`, and the documented 15 recommendation scenarios.

## Why random exposure matters

Random exposure is useful because recommender logs suffer from exposure bias and selection bias: observed feedback is conditional on what a previous policy chose to show. Randomized exposure can support debiasing, off-policy evaluation, and causal recommendation analysis. In this EDA we describe associations and distribution differences; we do not claim that random exposure caused a behavioral change without a task-specific causal design.


## Interaction Schema

The schema table below documents the expected interaction columns, semantic meaning, and possible modeling role. Actual dtypes are checked from local files in the next sections.


In [ ]:
interaction_schema = pd.DataFrame([
    {"column": "user_id", "semantic meaning": "Anonymized user identifier", "potential modeling role": "User key, grouping key for sequences"},
    {"column": "video_id", "semantic meaning": "Anonymized video/item identifier", "potential modeling role": "Item key, target item for ranking"},
    {"column": "date", "semantic meaning": "Calendar date of exposure/interaction", "potential modeling role": "Temporal split, trend analysis"},
    {"column": "hourmin", "semantic meaning": "Hour-minute value of interaction", "potential modeling role": "Time-of-day feature after careful validation"},
    {"column": "time_ms", "semantic meaning": "Timestamp in milliseconds", "potential modeling role": "Sequence ordering, chronological split"},
    {"column": "is_click", "semantic meaning": "Click feedback", "potential modeling role": "Binary target or auxiliary feedback"},
    {"column": "is_like", "semantic meaning": "Like feedback", "potential modeling role": "Sparse positive target or auxiliary task"},
    {"column": "is_follow", "semantic meaning": "Follow feedback", "potential modeling role": "Sparse engagement target"},
    {"column": "is_comment", "semantic meaning": "Comment feedback", "potential modeling role": "Sparse engagement target"},
    {"column": "is_forward", "semantic meaning": "Forward/share feedback", "potential modeling role": "Sparse engagement target"},
    {"column": "is_hate", "semantic meaning": "Negative feedback", "potential modeling role": "Negative preference / safety signal"},
    {"column": "long_view", "semantic meaning": "Long-view indicator", "potential modeling role": "Binary watch-time target"},
    {"column": "play_time_ms", "semantic meaning": "Observed play time", "potential modeling role": "Continuous engagement signal"},
    {"column": "duration_ms", "semantic meaning": "Video duration", "potential modeling role": "Normalization denominator; item-side feature with caution"},
    {"column": "profile_stay_time", "semantic meaning": "Time spent on profile", "potential modeling role": "Auxiliary engagement signal"},
    {"column": "comment_stay_time", "semantic meaning": "Time spent in comments", "potential modeling role": "Auxiliary engagement signal"},
    {"column": "is_profile_enter", "semantic meaning": "Profile-enter indicator", "potential modeling role": "Binary auxiliary feedback"},
    {"column": "is_rand", "semantic meaning": "Random-exposure flag", "potential modeling role": "Policy indicator; do not mix blindly"},
    {"column": "tab", "semantic meaning": "Recommendation scenario/tab", "potential modeling role": "Scenario/context feature or stratification key"},
])
interaction_schema["dtype"] = "checked from files below"
display(interaction_schema[["column", "dtype", "semantic meaning", "potential modeling role"]])


## Notebook Helpers

The helper functions below keep data access lazy and guard every data-dependent section. Pure and 1K may be loaded interactively after aggregation. 27K should be handled by `src/eda_27k.py` rather than full scans in this notebook.


In [ ]:
LOG_META = {
    "standard_early": {"policy": "standard", "label": "log_standard_4_08_to_4_21"},
    "standard_late": {"policy": "standard", "label": "log_standard_4_22_to_5_08"},
    "random": {"policy": "random", "label": "log_random_4_22_to_5_08"},
}

MAX_PLOT_ROWS = 200_000


def version_ready(version: str) -> bool:
    return bool(VERSION_ROOTS.get(version, Path("missing")).exists())


def skip(reason: str) -> None:
    print(f"Skipped: {reason}")


def pl_df_to_pandas(frame):
    return pd.DataFrame(frame.to_dicts())


def schema_records(lf) -> list[dict[str, str]]:
    if hasattr(lf, "collect_schema"):
        schema = lf.collect_schema()
        return [{"column": name, "dtype": str(schema[name])} for name in schema.names()]
    return [{"column": name, "dtype": str(dtype)} for name, dtype in lf.schema.items()]


def scan_interaction_log(version: str, log_key: str):
    if not version_ready(version):
        return None
    path = DISCOVERED_FILES.get(version, {}).get(log_key)
    if path is None:
        return None
    pl = require_polars()
    meta = LOG_META[log_key]
    return scan_table(path).with_columns([
        pl.lit(meta["policy"]).alias("policy"),
        pl.lit(meta["label"]).alias("source_log"),
    ])


def combine_interactions(version: str, policies: tuple[str, ...] = ("standard", "random")):
    if not version_ready(version):
        return None
    scans = []
    for log_key, meta in LOG_META.items():
        if meta["policy"] not in policies:
            continue
        lf = scan_interaction_log(version, log_key)
        if lf is not None:
            scans.append(lf)
    if not scans:
        return None
    return concat_lazy_frames(scans)


def table_or_message(df: pd.DataFrame, message: str) -> None:
    if df.empty:
        print(message)
    else:
        display(df)


## Actual Interaction Dtypes

This section checks actual dtypes from file schemas. It does not materialize the full datasets.


In [ ]:
schema_rows = []
for version in ("Pure", "1K"):
    if not version_ready(version):
        continue
    for log_key, meta in LOG_META.items():
        path = DISCOVERED_FILES.get(version, {}).get(log_key)
        if path is None:
            continue
        lf = scan_table(path)
        for row in schema_records(lf):
            schema_rows.append({
                "version": version,
                "log": meta["label"],
                **row,
            })

schema_df = pd.DataFrame(schema_rows)
table_or_message(schema_df, "No Pure/1K interaction schemas available to inspect.")


## Data Quality: Pure and 1K

For Pure and 1K this section computes rows, users, videos, duplicate rows, missing cells, timestamp/date ranges, `tab` values, and `is_rand` values. These values are **Computed from local files** only when the files exist.


In [ ]:
def interaction_quality(version: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if not version_ready(version):
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    pl = require_polars()
    quality_rows = []
    missing_rows = []
    binary_rows = []

    for log_key, meta in LOG_META.items():
        path = DISCOVERED_FILES.get(version, {}).get(log_key)
        if path is None:
            continue
        lf = scan_table(path)
        names = lazy_schema_names(lf)
        name_set = set(names)

        exprs = [pl.len().alias("rows")]
        if names:
            exprs.append(pl.struct(names).is_duplicated().sum().alias("duplicates"))
            exprs.append(
                pl.sum_horizontal([pl.col(col).is_null().cast(pl.Int64) for col in names]).sum().alias("missing_cells")
            )
        if "user_id" in name_set:
            exprs.append(pl.col("user_id").n_unique().alias("users"))
        if "video_id" in name_set:
            exprs.append(pl.col("video_id").n_unique().alias("videos"))
        if "time_ms" in name_set:
            exprs.extend([
                pl.col("time_ms").cast(pl.Int64, strict=False).min().alias("time_ms_min"),
                pl.col("time_ms").cast(pl.Int64, strict=False).max().alias("time_ms_max"),
            ])
        if "date" in name_set:
            exprs.extend([pl.col("date").min().alias("date_min"), pl.col("date").max().alias("date_max")])
        if "tab" in name_set:
            exprs.append(pl.col("tab").n_unique().alias("tab_n_values"))
        if "is_rand" in name_set:
            exprs.append(pl.col("is_rand").n_unique().alias("is_rand_n_values"))

        quality = collect_lazy(lf.select(exprs)).to_dicts()[0]
        quality_rows.append({
            "version": version,
            "log": meta["label"],
            "policy": meta["policy"],
            "source": "Computed from local files",
            **quality,
        })

        missing = collect_lazy(lf.select([pl.col(col).is_null().sum().alias(col) for col in names])).to_dicts()[0]
        for column, missing_count in missing.items():
            if missing_count:
                missing_rows.append({
                    "version": version,
                    "log": meta["label"],
                    "column": column,
                    "missing_count": missing_count,
                    "source": "Computed from local files",
                })

        for column in [col for col in FEEDBACK_COLUMNS if col in name_set]:
            value = pl.col(column).cast(pl.Int64, strict=False)
            binary_quality = collect_lazy(lf.select([
                value.is_null().sum().alias("missing_count"),
                value.n_unique().alias("unique_values"),
                (value.is_not_null() & (~value.is_in([0, 1]))).sum().alias("invalid_count"),
                value.min().alias("min_value"),
                value.max().alias("max_value"),
            ])).to_dicts()[0]
            binary_rows.append({
                "version": version,
                "log": meta["label"],
                "column": column,
                **binary_quality,
                "source": "Computed from local files",
            })

    return pd.DataFrame(quality_rows), pd.DataFrame(missing_rows), pd.DataFrame(binary_rows)


quality_tables = {}
missing_tables = {}
binary_tables = {}
for version in ("Pure", "1K"):
    quality, missing, binary = interaction_quality(version)
    quality_tables[version] = quality
    missing_tables[version] = missing
    binary_tables[version] = binary

quality_df = pd.concat([df for df in quality_tables.values() if not df.empty], ignore_index=True) if any(not df.empty for df in quality_tables.values()) else pd.DataFrame()
missing_df = pd.concat([df for df in missing_tables.values() if not df.empty], ignore_index=True) if any(not df.empty for df in missing_tables.values()) else pd.DataFrame()
binary_quality_df = pd.concat([df for df in binary_tables.values() if not df.empty], ignore_index=True) if any(not df.empty for df in binary_tables.values()) else pd.DataFrame()

table_or_message(quality_df, "No Pure/1K data-quality summary computed.")
table_or_message(missing_df, "No missing values found or no data available.")
table_or_message(binary_quality_df, "No binary feedback quality summary computed.")


In [ ]:
def distinct_values_for_columns(version: str, columns: tuple[str, ...] = ("tab", "is_rand")) -> pd.DataFrame:
    if not version_ready(version):
        return pd.DataFrame()
    pl = require_polars()
    rows = []
    for log_key, meta in LOG_META.items():
        path = DISCOVERED_FILES.get(version, {}).get(log_key)
        if path is None:
            continue
        lf = scan_table(path)
        names = set(lazy_schema_names(lf))
        for column in columns:
            if column not in names:
                continue
            values = collect_lazy(lf.select(pl.col(column).drop_nulls().unique().sort()).limit(100)).to_dicts()
            rows.append({
                "version": version,
                "log": meta["label"],
                "column": column,
                "values_preview": [row[column] for row in values],
                "source": "Computed from local files",
            })
    return pd.DataFrame(rows)

value_previews = pd.concat(
    [distinct_values_for_columns(version) for version in ("Pure", "1K")],
    ignore_index=True,
) if any(version_ready(version) for version in ("Pure", "1K")) else pd.DataFrame()
table_or_message(value_previews, "No tab/is_rand value previews computed.")


## Standard vs Random Counts

This section computes `Version`, `Policy`, `Users`, `Items`, and `Interactions` for Pure and 1K. 27K is intentionally excluded from this notebook to avoid heavy interactive scans.


In [ ]:
def policy_counts_for_version(version: str) -> pd.DataFrame:
    lf = combine_interactions(version)
    if lf is None:
        return pd.DataFrame()
    pl = require_polars()
    names = set(lazy_schema_names(lf))
    exprs = [pl.len().alias("Interactions")]
    if "user_id" in names:
        exprs.append(pl.col("user_id").n_unique().alias("Users"))
    if "video_id" in names:
        exprs.append(pl.col("video_id").n_unique().alias("Items"))
    records = collect_lazy(lf.group_by("policy").agg(exprs).sort("policy")).to_dicts()
    df = pd.DataFrame(records).rename(columns={"policy": "Policy"})
    if df.empty:
        return df
    df.insert(0, "Version", version)
    total = df["Interactions"].sum()
    df["interaction_share"] = df["Interactions"] / total if total else np.nan
    df["source"] = "Computed from local files"
    return df

policy_counts_df = pd.concat(
    [policy_counts_for_version(version) for version in ("Pure", "1K")],
    ignore_index=True,
) if any(version_ready(version) for version in ("Pure", "1K")) else pd.DataFrame()
table_or_message(policy_counts_df, "No standard-vs-random counts computed for Pure/1K.")


## Feedback Signals

The feedback signals analyzed here are `is_click`, `is_like`, `is_follow`, `is_comment`, `is_forward`, `is_hate`, `long_view`, and `is_profile_enter`. Counts and rates are computed separately for standard and random exposure.


In [ ]:
def feedback_by_policy(version: str) -> pd.DataFrame:
    lf = combine_interactions(version)
    if lf is None:
        return pd.DataFrame()
    pl = require_polars()
    feedback_cols = available_columns(lf, FEEDBACK_COLUMNS)
    if not feedback_cols:
        return pd.DataFrame()

    exprs = []
    for column in feedback_cols:
        value = pl.col(column).cast(pl.Float64, strict=False)
        exprs.extend([
            value.sum().alias(f"{column}__positive_count"),
            value.mean().alias(f"{column}__positive_rate"),
            pl.col(column).is_null().sum().alias(f"{column}__missing_count"),
        ])
    wide = collect_lazy(lf.group_by("policy").agg(exprs).sort("policy")).to_dicts()

    rows = []
    for row in wide:
        for column in feedback_cols:
            rows.append({
                "version": version,
                "policy": row["policy"],
                "signal": column,
                "positive_count": row.get(f"{column}__positive_count"),
                "positive_rate": row.get(f"{column}__positive_rate"),
                "missing_count": row.get(f"{column}__missing_count"),
                "source": "Computed from local files",
            })
    return pd.DataFrame(rows)

feedback_rates_df = pd.concat(
    [feedback_by_policy(version) for version in ("Pure", "1K")],
    ignore_index=True,
) if any(version_ready(version) for version in ("Pure", "1K")) else pd.DataFrame()
table_or_message(feedback_rates_df, "No feedback-rate table computed.")


In [ ]:
if feedback_rates_df.empty:
    skip("feedback-rate plot requires Pure or 1K data")
else:
    plot_version = "1K" if "1K" in set(feedback_rates_df["version"]) else feedback_rates_df["version"].iloc[0]
    pivot = feedback_rates_df[feedback_rates_df["version"] == plot_version].pivot(
        index="signal", columns="policy", values="positive_rate"
    )
    ax = pivot.plot(kind="bar", figsize=(10, 4), rot=35)
    ax.set_title(f"Positive feedback rate by policy: KuaiRand-{plot_version}")
    ax.set_ylabel("positive rate")
    ax.set_xlabel("")
    plt.tight_layout()
    plt.show()


## Watch-Time Analysis

`play_ratio = play_time_ms / duration_ms` is created only in memory. Rows with `duration_ms <= 0` receive null `play_ratio`. Clipping is used only for visualization and never changes raw values.


In [ ]:
def with_play_ratio(lf):
    pl = require_polars()
    names = set(lazy_schema_names(lf))
    if not {"play_time_ms", "duration_ms"}.issubset(names):
        return lf
    return lf.with_columns(
        pl.when(pl.col("duration_ms").cast(pl.Float64, strict=False) > 0)
        .then(pl.col("play_time_ms").cast(pl.Float64, strict=False) / pl.col("duration_ms").cast(pl.Float64, strict=False))
        .otherwise(None)
        .alias("play_ratio")
    )


def watch_time_percentiles(version: str) -> pd.DataFrame:
    lf = combine_interactions(version)
    if lf is None:
        return pd.DataFrame()
    pl = require_polars()
    lf = with_play_ratio(lf)
    names = set(lazy_schema_names(lf))
    cols = [col for col in ("play_time_ms", "duration_ms", "play_ratio") if col in names]
    if not cols:
        return pd.DataFrame()

    exprs = []
    for column in cols:
        value = pl.col(column).cast(pl.Float64, strict=False)
        exprs.extend([
            value.min().alias(f"{column}_min"),
            value.quantile(0.25).alias(f"{column}_p25"),
            value.quantile(0.50).alias(f"{column}_p50"),
            value.quantile(0.75).alias(f"{column}_p75"),
            value.quantile(0.90).alias(f"{column}_p90"),
            value.quantile(0.95).alias(f"{column}_p95"),
            value.quantile(0.99).alias(f"{column}_p99"),
            value.max().alias(f"{column}_max"),
        ])
    records = collect_lazy(lf.group_by("policy").agg(exprs).sort("policy")).to_dicts()
    df = pd.DataFrame(records)
    if not df.empty:
        df.insert(0, "version", version)
        df["source"] = "Computed from local files; raw percentiles before visualization clipping"
    return df

watch_time_summary_df = pd.concat(
    [watch_time_percentiles(version) for version in ("Pure", "1K")],
    ignore_index=True,
) if any(version_ready(version) for version in ("Pure", "1K")) else pd.DataFrame()
table_or_message(watch_time_summary_df, "No watch-time percentiles computed.")


In [ ]:
def watch_time_visual_sample(version: str, limit: int = MAX_PLOT_ROWS) -> pd.DataFrame:
    lf = combine_interactions(version)
    if lf is None:
        return pd.DataFrame()
    lf = with_play_ratio(lf)
    names = set(lazy_schema_names(lf))
    columns = [col for col in ["policy", "play_time_ms", "duration_ms", "play_ratio", "is_click", "long_view"] if col in names]
    if not columns:
        return pd.DataFrame()
    records = collect_lazy(lf.select(columns).drop_nulls(subset=["play_ratio"]).limit(limit)).to_dicts()
    return pd.DataFrame(records)

plot_version = "1K" if version_ready("1K") else "Pure"
watch_sample_df = watch_time_visual_sample(plot_version) if version_ready(plot_version) else pd.DataFrame()
if watch_sample_df.empty:
    skip("watch-time plots require Pure or 1K interaction logs")
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    watch_sample_df["duration_ms"].clip(upper=watch_sample_df["duration_ms"].quantile(0.99)).hist(ax=axes[0], bins=60)
    axes[0].set_title("Duration distribution, p99-clipped for plot")
    axes[0].set_xlabel("duration_ms")

    watch_sample_df["play_time_ms"].clip(upper=watch_sample_df["play_time_ms"].quantile(0.99)).hist(ax=axes[1], bins=60)
    axes[1].set_title("Play-time distribution, p99-clipped for plot")
    axes[1].set_xlabel("play_time_ms")

    watch_sample_df["play_ratio"].clip(upper=watch_sample_df["play_ratio"].quantile(0.99)).hist(ax=axes[2], bins=60)
    axes[2].set_title("Play ratio, p99-clipped for plot")
    axes[2].set_xlabel("play_ratio")
    plt.tight_layout()
    plt.show()

    for target in [col for col in ("is_click", "long_view") if col in watch_sample_df.columns]:
        ax = watch_sample_df.assign(
            play_ratio_plot=lambda df: df["play_ratio"].clip(upper=df["play_ratio"].quantile(0.99))
        ).boxplot(column="play_ratio_plot", by=target, figsize=(6, 4))
        ax.set_title(f"Play ratio vs {target}; visualization clipped")
        ax.set_xlabel(target)
        ax.set_ylabel("play_ratio")
        plt.suptitle("")
        plt.tight_layout()
        plt.show()


## Distribution Shift: Standard vs Random

This section compares standard-policy and random-exposure logs using feedback rates, watch-time summaries, unique item counts, and interactions per user. Interpret differences as associations / distribution differences, not as causal effects.


In [ ]:
def distribution_shift_summary(version: str) -> pd.DataFrame:
    lf = combine_interactions(version)
    if lf is None:
        return pd.DataFrame()
    pl = require_polars()
    lf = with_play_ratio(lf)
    names = set(lazy_schema_names(lf))

    exprs = [pl.len().alias("interactions")]
    if "user_id" in names:
        exprs.append(pl.col("user_id").n_unique().alias("users"))
    if "video_id" in names:
        exprs.append(pl.col("video_id").n_unique().alias("unique_items"))
    for column in ["is_click", "long_view", "is_like", "is_follow", "is_comment"]:
        if column in names:
            exprs.append(pl.col(column).cast(pl.Float64, strict=False).mean().alias(f"{column}_rate"))
    for column in ["play_time_ms", "duration_ms", "play_ratio"]:
        if column in names:
            exprs.append(pl.col(column).cast(pl.Float64, strict=False).mean().alias(f"{column}_mean"))
            exprs.append(pl.col(column).cast(pl.Float64, strict=False).quantile(0.5).alias(f"{column}_median"))

    base = pd.DataFrame(collect_lazy(lf.group_by("policy").agg(exprs).sort("policy")).to_dicts())
    if base.empty:
        return base

    if "user_id" in names:
        per_user = pd.DataFrame(collect_lazy(
            lf.group_by(["policy", "user_id"]).agg(pl.len().alias("interactions_per_user"))
            .group_by("policy")
            .agg([
                pl.col("interactions_per_user").mean().alias("interactions_per_user_mean"),
                pl.col("interactions_per_user").quantile(0.5).alias("interactions_per_user_median"),
                pl.col("interactions_per_user").quantile(0.9).alias("interactions_per_user_p90"),
            ])
            .sort("policy")
        ).to_dicts())
        base = base.merge(per_user, on="policy", how="left")

    base.insert(0, "version", version)
    base["source"] = "Computed from local files"
    return base

shift_summary_df = pd.concat(
    [distribution_shift_summary(version) for version in ("Pure", "1K")],
    ignore_index=True,
) if any(version_ready(version) for version in ("Pure", "1K")) else pd.DataFrame()
table_or_message(shift_summary_df, "No distribution-shift summary computed.")


In [ ]:
if shift_summary_df.empty:
    skip("distribution-shift plots require computed summary tables")
else:
    plot_version = "1K" if "1K" in set(shift_summary_df["version"]) else shift_summary_df["version"].iloc[0]
    plot_df = shift_summary_df[shift_summary_df["version"] == plot_version].set_index("policy")

    rate_cols = [col for col in plot_df.columns if col.endswith("_rate")]
    if rate_cols:
        ax = plot_df[rate_cols].T.plot(kind="bar", figsize=(10, 4), rot=35)
        ax.set_title(f"Feedback-rate differences by policy: KuaiRand-{plot_version}")
        ax.set_ylabel("rate")
        plt.tight_layout()
        plt.show()

    mean_cols = [col for col in ["play_time_ms_mean", "duration_ms_mean", "play_ratio_mean"] if col in plot_df.columns]
    if mean_cols:
        ax = plot_df[mean_cols].T.plot(kind="bar", figsize=(8, 4), rot=35)
        ax.set_title(f"Watch-time distribution differences: KuaiRand-{plot_version}")
        ax.set_ylabel("mean value")
        plt.tight_layout()
        plt.show()

    if "interactions_per_user_mean" in plot_df.columns:
        ax = plot_df[["interactions_per_user_mean", "interactions_per_user_median", "interactions_per_user_p90"]].T.plot(
            kind="bar", figsize=(8, 4), rot=35
        )
        ax.set_title(f"Interactions per user by policy: KuaiRand-{plot_version}")
        ax.set_ylabel("interactions")
        plt.tight_layout()
        plt.show()


## Tab Analysis

`tab` represents recommendation scenario/context. It should be inspected before using it as a feature, filter, or evaluation stratum.


In [ ]:
def tab_analysis(version: str) -> pd.DataFrame:
    lf = combine_interactions(version)
    if lf is None:
        return pd.DataFrame()
    pl = require_polars()
    names = set(lazy_schema_names(lf))
    if "tab" not in names:
        return pd.DataFrame()
    exprs = [pl.len().alias("interactions")]
    if "user_id" in names:
        exprs.append(pl.col("user_id").n_unique().alias("users"))
    if "video_id" in names:
        exprs.append(pl.col("video_id").n_unique().alias("items"))
    for column in ["is_click", "long_view", "is_like"]:
        if column in names:
            exprs.append(pl.col(column).cast(pl.Float64, strict=False).mean().alias(f"{column}_rate"))

    df = pd.DataFrame(collect_lazy(lf.group_by("tab").agg(exprs).sort("interactions", descending=True)).to_dicts())
    if df.empty:
        return df
    df.insert(0, "version", version)
    df["share"] = df["interactions"] / df["interactions"].sum()
    df["source"] = "Computed from local files"
    return df

tab_df = tab_analysis("1K") if version_ready("1K") else tab_analysis("Pure")
table_or_message(tab_df, "No tab analysis computed.")


In [ ]:
if tab_df.empty:
    skip("tab distribution plot requires computed tab table")
else:
    ax = tab_df.sort_values("interactions", ascending=False).plot(
        x="tab", y="interactions", kind="bar", figsize=(10, 4), legend=False, rot=35
    )
    ax.set_title("Distribution of interactions across tab")
    ax.set_xlabel("tab")
    ax.set_ylabel("interactions")
    plt.tight_layout()
    plt.show()


## User Features

The expected user features include activity degree, low-activity period, creator/live-streamer flags, follow/fans/friend counts and ranges, registration age, and encrypted categorical columns named `onehot_feat*`.

`onehot_feat*` columns should be treated as encrypted/categorical features. Their labels are not directly interpretable, but cardinality and missingness still matter for modeling.


In [ ]:
USER_FEATURE_COLUMNS = (
    "user_active_degree",
    "is_lowactive_period",
    "is_live_streamer",
    "is_video_author",
    "follow_user_num",
    "follow_user_num_range",
    "fans_user_num",
    "fans_user_num_range",
    "friend_user_num",
    "friend_user_num_range",
    "register_days",
    "register_days_range",
)


def feature_table_path(version: str, key: str) -> Path | None:
    if not version_ready(version):
        return None
    return DISCOVERED_FILES.get(version, {}).get(key)


def table_shape_and_schema(path: Path) -> tuple[dict[str, int], pd.DataFrame]:
    pl = require_polars()
    lf = scan_table(path)
    row_count = collect_lazy(lf.select(pl.len().alias("rows"))).to_dicts()[0]["rows"]
    schema = pd.DataFrame(schema_records(lf))
    return {"rows": row_count, "columns": len(schema)}, schema


def missing_and_cardinality(path: Path, columns: list[str]) -> pd.DataFrame:
    pl = require_polars()
    lf = scan_table(path)
    names = set(lazy_schema_names(lf))
    selected = [column for column in columns if column in names]
    if not selected:
        return pd.DataFrame()
    exprs = []
    for column in selected:
        exprs.extend([
            pl.col(column).is_null().sum().alias(f"{column}__missing"),
            pl.col(column).n_unique().alias(f"{column}__cardinality"),
        ])
    row = collect_lazy(lf.select(exprs)).to_dicts()[0]
    records = []
    for column in selected:
        records.append({
            "column": column,
            "missing": row.get(f"{column}__missing"),
            "cardinality": row.get(f"{column}__cardinality"),
            "source": "Computed from local files",
        })
    return pd.DataFrame(records)


def top_values(path: Path, column: str, limit: int = 20) -> pd.DataFrame:
    pl = require_polars()
    lf = scan_table(path)
    if column not in set(lazy_schema_names(lf)):
        return pd.DataFrame()
    records = collect_lazy(
        lf.group_by(column).agg(pl.len().alias("rows")).sort("rows", descending=True).limit(limit)
    ).to_dicts()
    return pd.DataFrame(records)


feature_version = "1K" if version_ready("1K") else "Pure"
user_path = feature_table_path(feature_version, "user_features")
if user_path is None:
    skip("user feature table is unavailable")
else:
    shape, user_schema_df = table_shape_and_schema(user_path)
    display(pd.DataFrame([{**shape, "version": feature_version, "path": str(user_path), "source": "Computed from local files"}]))
    display(user_schema_df)

    onehot_columns = [row["column"] for row in user_schema_df.to_dict("records") if row["column"].startswith("onehot_feat")]
    user_missing_cardinality = missing_and_cardinality(user_path, list(USER_FEATURE_COLUMNS) + onehot_columns)
    display(user_missing_cardinality)

    for column in ["user_active_degree", "follow_user_num_range", "fans_user_num_range", "register_days_range"]:
        tv = top_values(user_path, column)
        if not tv.empty:
            display(Markdown(f"### Top values: `{column}`"))
            display(tv)


## Video Basic Features

Video basic features describe item metadata such as author, type, upload information, visibility, dimensions, duration, music, and tags. These are item-side features, but their availability and timestamp semantics should be checked before modeling.


In [ ]:
VIDEO_BASIC_COLUMNS = (
    "author_id",
    "video_type",
    "upload_dt",
    "upload_type",
    "visible_status",
    "video_duration",
    "server_width",
    "server_height",
    "music_id",
    "music_type",
    "tag",
)


def numeric_column_summary_from_scan(path: Path, columns: list[str]) -> pd.DataFrame:
    pl = require_polars()
    lf = scan_table(path)
    names = set(lazy_schema_names(lf))
    selected = [column for column in columns if column in names]
    if not selected:
        return pd.DataFrame()
    exprs = []
    for column in selected:
        value = pl.col(column).cast(pl.Float64, strict=False)
        exprs.extend([
            value.count().alias(f"{column}__count"),
            value.min().alias(f"{column}__min"),
            value.mean().alias(f"{column}__mean"),
            value.quantile(0.5).alias(f"{column}__median"),
            value.quantile(0.9).alias(f"{column}__p90"),
            value.quantile(0.99).alias(f"{column}__p99"),
            value.max().alias(f"{column}__max"),
        ])
    row = collect_lazy(lf.select(exprs)).to_dicts()[0]
    records = []
    for column in selected:
        records.append({
            "column": column,
            "count": row.get(f"{column}__count"),
            "min": row.get(f"{column}__min"),
            "mean": row.get(f"{column}__mean"),
            "median": row.get(f"{column}__median"),
            "p90": row.get(f"{column}__p90"),
            "p99": row.get(f"{column}__p99"),
            "max": row.get(f"{column}__max"),
            "source": "Computed from local files",
        })
    return pd.DataFrame(records)


video_basic_path = feature_table_path(feature_version, "video_basic")
if video_basic_path is None:
    skip("video basic feature table is unavailable")
else:
    shape, video_basic_schema_df = table_shape_and_schema(video_basic_path)
    display(pd.DataFrame([{**shape, "version": feature_version, "path": str(video_basic_path), "source": "Computed from local files"}]))
    display(video_basic_schema_df)

    for column in ["video_type", "upload_type", "visible_status", "music_type"]:
        tv = top_values(video_basic_path, column)
        if not tv.empty:
            display(Markdown(f"### Distribution: `{column}`"))
            display(tv)

    duration_summary = numeric_column_summary_from_scan(video_basic_path, ["video_duration", "server_width", "server_height"])
    table_or_message(duration_summary, "No numeric video-basic summary computed.")

    if "tag" in set(video_basic_schema_df["column"]):
        tag_records = collect_lazy(scan_table(video_basic_path).select("tag").drop_nulls().limit(200_000)).to_dicts()
        tag_values = [str(row["tag"]) for row in tag_records]
        tag_ids = sorted({token for value in tag_values for token in re.findall(r"\\d+", value)})
        display(pd.DataFrame([{
            "rows_scanned": len(tag_values),
            "approx_unique_tag_ids_in_scanned_rows": len(tag_ids),
            "tag_examples": tag_values[:10],
            "tag_id_examples": tag_ids[:20],
            "source": "Computed from local files; sample-limited approximate tag parsing",
        }]))


## Video Statistic Features

Video statistic features such as `show_cnt`, `play_cnt`, `play_duration`, `complete_play_cnt`, `valid_play_cnt`, `long_time_play_cnt`, `short_time_play_cnt`, `play_progress`, `like_cnt`, `comment_cnt`, `follow_cnt`, `share_cnt`, and `collect_cnt` are aggregated video statistics. They are not individual interaction records.


In [ ]:
VIDEO_STAT_COLUMNS = (
    "show_cnt",
    "play_cnt",
    "play_duration",
    "complete_play_cnt",
    "valid_play_cnt",
    "long_time_play_cnt",
    "short_time_play_cnt",
    "play_progress",
    "like_cnt",
    "comment_cnt",
    "follow_cnt",
    "share_cnt",
    "collect_cnt",
)

video_stats_version = "Pure" if version_ready("Pure") else feature_version
video_stats_path = feature_table_path(video_stats_version, "video_statistics")
if video_stats_path is None:
    skip("video statistic feature table is unavailable")
else:
    shape, video_stats_schema_df = table_shape_and_schema(video_stats_path)
    display(pd.DataFrame([{**shape, "version": video_stats_version, "path": str(video_stats_path), "source": "Computed from local files"}]))
    display(video_stats_schema_df)

    video_stat_summary = numeric_column_summary_from_scan(video_stats_path, list(VIDEO_STAT_COLUMNS))
    table_or_message(video_stat_summary, "No numeric video-statistic summary computed.")

    if "show_cnt" in set(video_stats_schema_df["column"]):
        display(Markdown("### Top videos by aggregated show count"))
        display(pd.DataFrame(collect_lazy(
            scan_table(video_stats_path)
            .sort("show_cnt", descending=True)
            .select([col for col in ["video_id", "show_cnt", "play_cnt", "like_cnt", "comment_cnt"] if col in set(video_stats_schema_df["column"])])
            .limit(20)
        ).to_dicts()))


## Leakage Warning

Video statistic features may contain information aggregated over a period that is later than a particular training interaction. They must not be automatically passed to a model without checking their construction time window.

### Potential leakage checklist

1. Future video statistics.
2. Popularity calculated using future interactions.
3. Filtering before temporal split.
4. Mixing random and standard logs without policy-aware design.
5. Target-derived statistics.
6. Preprocessing fitted on the full dataset before train-only fitting.
7. Temporal leakage through timestamps, upload dates, or derived windows.
8. Item/user eligibility determined using future information.


## Sequential Structure

The main sequential analysis is performed on KuaiRand-1K by combining the standard logs and ordering each user's interactions by `time_ms`.


In [ ]:
def sequence_analysis_1k() -> tuple[pd.DataFrame, pd.DataFrame]:
    lf = combine_interactions("1K", policies=("standard",))
    if lf is None:
        return pd.DataFrame(), pd.DataFrame()
    pl = require_polars()
    names = set(lazy_schema_names(lf))
    if "user_id" not in names:
        return pd.DataFrame(), pd.DataFrame()

    exprs = [pl.len().alias("interactions")]
    if "time_ms" in names:
        time_col = pl.col("time_ms").cast(pl.Int64, strict=False)
        exprs.extend([time_col.min().alias("first_timestamp"), time_col.max().alias("last_timestamp")])
    if "video_id" in names:
        exprs.append(pl.col("video_id").n_unique().alias("unique_videos"))

    per_user = pd.DataFrame(collect_lazy(lf.group_by("user_id").agg(exprs).sort("user_id")).to_dicts())
    if per_user.empty:
        return pd.DataFrame(), pd.DataFrame()

    summary = pd.DataFrame([{**numeric_summary(per_user["interactions"].to_numpy()), "source": "Computed from local files"}])
    sample_positions = np.linspace(0, len(per_user) - 1, num=min(10, len(per_user)), dtype=int)
    deterministic_users = per_user.iloc[sample_positions].copy()
    return summary, deterministic_users

sequence_summary_df, deterministic_users_df = sequence_analysis_1k()
table_or_message(sequence_summary_df, "No 1K sequence-length summary computed.")
table_or_message(deterministic_users_df, "No deterministic user sequence sample computed.")


In [ ]:
if deterministic_users_df.empty:
    skip("sequence-length plot requires 1K standard logs")
else:
    lf_1k_standard = combine_interactions("1K", policies=("standard",))
    pl = require_polars()
    per_user_counts = pd.DataFrame(collect_lazy(
        lf_1k_standard.group_by("user_id").agg(pl.len().alias("interactions"))
    ).to_dicts())
    ax = per_user_counts["interactions"].hist(bins=60, figsize=(8, 4))
    ax.set_title("1K standard sequence length distribution")
    ax.set_xlabel("interactions per user")
    ax.set_ylabel("users")
    ax.set_yscale("log")
    plt.tight_layout()
    plt.show()


## Sparsity and Long Tail

On KuaiRand-1K standard logs, this section summarizes interactions per user, unique videos per user, interactions per video, and unique users per video. Approximate matrix sparsity is computed over the observed user and item universe in the selected logs.


In [ ]:
def sparsity_long_tail_1k() -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    lf = combine_interactions("1K", policies=("standard",))
    if lf is None:
        return pd.DataFrame(), {}
    pl = require_polars()
    names = set(lazy_schema_names(lf))
    if not {"user_id", "video_id"}.issubset(names):
        return pd.DataFrame(), {}

    user_items = pd.DataFrame(collect_lazy(
        lf.group_by("user_id").agg([
            pl.len().alias("interactions_per_user"),
            pl.col("video_id").n_unique().alias("unique_videos_per_user"),
        ])
    ).to_dicts())
    item_users = pd.DataFrame(collect_lazy(
        lf.group_by("video_id").agg([
            pl.len().alias("interactions_per_video"),
            pl.col("user_id").n_unique().alias("unique_users_per_video"),
        ])
    ).to_dicts())
    observed_pairs = collect_lazy(lf.select(["user_id", "video_id"]).unique().select(pl.len().alias("pairs"))).to_dicts()[0]["pairs"]
    n_users = len(user_items)
    n_items = len(item_users)
    sparsity = 1 - observed_pairs / (n_users * n_items) if n_users and n_items else np.nan

    summary_rows = []
    for label, frame, column in [
        ("interactions per user", user_items, "interactions_per_user"),
        ("unique videos per user", user_items, "unique_videos_per_user"),
        ("interactions per video", item_users, "interactions_per_video"),
        ("unique users per video", item_users, "unique_users_per_video"),
    ]:
        summary_rows.append({"quantity": label, **numeric_summary(frame[column].to_numpy()), "source": "Computed from local files"})
    summary_rows.append({
        "quantity": "interaction matrix sparsity over observed 1K standard user/item universe",
        "count": observed_pairs,
        "min": None,
        "mean": sparsity,
        "median": None,
        "p75": None,
        "p90": None,
        "p95": None,
        "p99": None,
        "max": None,
        "source": "Computed from local files",
    })
    return pd.DataFrame(summary_rows), {"user_items": user_items, "item_users": item_users}

sparsity_summary_df, long_tail_tables = sparsity_long_tail_1k()
table_or_message(sparsity_summary_df, "No sparsity/long-tail summary computed.")


In [ ]:
if not long_tail_tables:
    skip("long-tail plots require 1K standard logs")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    long_tail_tables["user_items"]["interactions_per_user"].hist(ax=axes[0], bins=60)
    axes[0].set_title("Interactions per user")
    axes[0].set_yscale("log")

    long_tail_tables["item_users"]["interactions_per_video"].clip(
        upper=long_tail_tables["item_users"]["interactions_per_video"].quantile(0.99)
    ).hist(ax=axes[1], bins=60)
    axes[1].set_title("Interactions per video, p99-clipped for plot")
    axes[1].set_yscale("log")
    plt.tight_layout()
    plt.show()


## Temporal EDA

This section converts the available date/timestamp fields carefully and counts interactions by day for standard and random logs. The documented collection windows are:

- 2022-04-08 to 2022-04-21
- 2022-04-22 to 2022-05-08


In [ ]:
def temporal_counts(version: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    lf = combine_interactions(version)
    if lf is None:
        return pd.DataFrame(), pd.DataFrame()
    pl = require_polars()
    names = set(lazy_schema_names(lf))
    daily = pd.DataFrame()
    hourly = pd.DataFrame()

    if "date" in names:
        daily = pd.DataFrame(collect_lazy(
            lf.group_by(["policy", "date"]).agg(pl.len().alias("interactions")).sort(["date", "policy"])
        ).to_dicts())
        if not daily.empty:
            daily.insert(0, "version", version)
            daily["source"] = "Computed from local files"

    if "hourmin" in names:
        hourly_lf = lf.with_columns((pl.col("hourmin").cast(pl.Int64, strict=False) // 100).alias("hour"))
        hourly = pd.DataFrame(collect_lazy(
            hourly_lf.group_by(["policy", "hour"]).agg(pl.len().alias("interactions")).sort(["policy", "hour"])
        ).to_dicts())
        if not hourly.empty:
            hourly.insert(0, "version", version)
            hourly["source"] = "Computed from local files"

    return daily, hourly

temporal_version = "1K" if version_ready("1K") else "Pure"
daily_df, hourly_df = temporal_counts(temporal_version) if version_ready(temporal_version) else (pd.DataFrame(), pd.DataFrame())
table_or_message(daily_df, "No daily temporal counts computed.")
table_or_message(hourly_df, "No hourly temporal counts computed.")


In [ ]:
if daily_df.empty:
    skip("daily timeline plot requires computed daily counts")
else:
    plot_daily = daily_df.copy()
    plot_daily["date_plot"] = pd.to_datetime(plot_daily["date"].astype(str), errors="coerce")
    pivot = plot_daily.pivot_table(index="date_plot", columns="policy", values="interactions", aggfunc="sum").sort_index()
    ax = pivot.plot(figsize=(11, 4), marker="o")
    ax.axvspan(pd.Timestamp("2022-04-08"), pd.Timestamp("2022-04-21"), alpha=0.12, color="tab:blue")
    ax.axvspan(pd.Timestamp("2022-04-22"), pd.Timestamp("2022-05-08"), alpha=0.12, color="tab:orange")
    ax.set_title(f"Interactions by day: KuaiRand-{temporal_version}")
    ax.set_xlabel("date")
    ax.set_ylabel("interactions")
    plt.tight_layout()
    plt.show()

if hourly_df.empty:
    skip("hourly plot requires computed hourly counts")
else:
    pivot = hourly_df.pivot_table(index="hour", columns="policy", values="interactions", aggfunc="sum").sort_index()
    ax = pivot.plot(kind="bar", figsize=(11, 4), rot=0)
    ax.set_title(f"Interactions by hour: KuaiRand-{temporal_version}")
    ax.set_xlabel("hour")
    ax.set_ylabel("interactions")
    plt.tight_layout()
    plt.show()


## Computed KuaiRand-27K Slurm Summary

If `outputs/eda/27k_summary.json` is present, this section loads compact results from the full cHARISMa Slurm run. These are **Computed from local files** results, not documentation values. The notebook does not load raw 27K interaction logs here.

In [ ]:
summary_27k_path = OUTPUT_DIR / "27k_summary.json"
if not summary_27k_path.exists():
    print(f"No computed 27K summary found at {summary_27k_path}")
else:
    summary_27k = json.loads(summary_27k_path.read_text(encoding="utf-8"))
    display(pd.DataFrame([{
        "mode": summary_27k.get("mode"),
        "generated_at_utc": summary_27k.get("generated_at_utc"),
        "version_root": summary_27k.get("version_root"),
        "disk_size": summary_27k.get("disk_size"),
        "source": "Computed from local files via cHARISMa Slurm",
    }]))

    for title, key in [
        ("Policy counts", "policy_counts"),
        ("Source counts", "source_counts"),
        ("Feedback summary", "feedback_summary"),
        ("Binary feedback quality", "binary_feedback_quality"),
        ("Watch-time summary", "watch_time_summary"),
        ("Sequence summary", "sequence_summary"),
        ("Item popularity summary", "item_popularity_summary"),
        ("Tab summary preview", "tab_summary_preview"),
    ]:
        records = summary_27k.get(key, [])
        if records:
            display(Markdown(f"### {title}"))
            display(pd.DataFrame(records))


## Pure vs 1K vs 27K

This comparison table combines documented dataset characteristics with disk size computed from local files when available. Sequential completeness, development usefulness, and full-experiment usefulness are methodological notes, not computed metrics.


In [ ]:
methodological_notes = pd.DataFrame([
    {"version": "KuaiRand-Pure", "Sequential completeness": "Candidate-pool-focused; useful for randomized/common-item analysis", "Development usefulness": "Good for quick policy and debiasing EDA", "Full experiment usefulness": "Useful when the task needs the Pure candidate pool"},
    {"version": "KuaiRand-1K", "Sequential completeness": "Dense per-user histories for 1,000 users", "Development usefulness": "Best interactive development version", "Full experiment usefulness": "Good for protocol debugging and baseline reproduction checks"},
    {"version": "KuaiRand-27K", "Sequential completeness": "Full large-scale user set", "Development usefulness": "Too heavy for interactive notebook scans", "Full experiment usefulness": "Main full-scale experiment candidate"},
])
version_key = {"KuaiRand-Pure": "Pure", "KuaiRand-1K": "1K", "KuaiRand-27K": "27K"}

final_comparison = documentation_stats.rename(columns={
    "users": "Users",
    "standard_items": "Items",
    "standard_interactions": "Standard interactions",
    "random_interactions": "Random interactions",
})[["version", "Users", "Items", "Standard interactions", "Random interactions", "source"]]
final_comparison["version_key"] = final_comparison["version"].map(version_key)
disk_for_merge = disk_summary[["version", "disk_size", "source"]].rename(
    columns={"version": "version_key", "source": "disk_size_source"}
)
final_comparison = final_comparison.merge(disk_for_merge, on="version_key", how="left")
final_comparison = final_comparison.merge(methodological_notes, on="version", how="left")
final_comparison = final_comparison.drop(columns=["version_key"])
display(final_comparison)


# Train / Validation / Test Protocols

KuaiRand raw release does not itself imply one universal train/validation/test protocol for every recommendation task.

Before running baselines, choose a published benchmark family and reproduce its preprocessing, split, candidate generation, negative sampling, filtering, and evaluation exactly.


In [ ]:
protocols = pd.DataFrame([
    {
        "Protocol": "Sequential leave-one-out",
        "Train": "For each user: i1 ... i(n-2)",
        "Validation": "i(n-1)",
        "Test": "i(n)",
        "Suitable tasks": "Next-item prediction, sequential recommendation",
        "Pros": "User-level temporal order; common in sequential recommenders",
        "Cons": "May ignore global calendar drift; sensitive to filtering and short histories",
        "Comparability concerns": "Candidate set, negative sampling, minimum history length, and repeated items must match published work",
    },
    {
        "Protocol": "Chronological/date split",
        "Train": "Early calendar period",
        "Validation": "Later validation window",
        "Test": "Final calendar window",
        "Suitable tasks": "Temporal generalization, production-like ranking",
        "Pros": "Respects global time and deployment chronology",
        "Cons": "Cold-start and changing item universe require explicit rules",
        "Comparability concerns": "Date boundaries, item/user eligibility, and feature time windows must be identical",
    },
    {
        "Protocol": "Standard training + random-exposure evaluation",
        "Train": "Standard-policy interactions",
        "Validation": "Task-specific: standard or held-out random exposure",
        "Test": "Random-exposure component when appropriate",
        "Suitable tasks": "Debiased recommendation, off-policy evaluation, causal/random-exposure evaluation",
        "Pros": "Uses randomized exposure to reduce policy-selection bias in evaluation",
        "Cons": "Random log is not automatically the right test set for every objective",
        "Comparability concerns": "Exposure policy, candidate pool, IPS/causal estimators, and target definition must match",
    },
])
display(protocols)


## Possible Tasks


In [ ]:
tasks = pd.DataFrame([
    {"Task": "next-item prediction", "Target": "next video_id", "Inputs": "ordered user history", "Recommended KuaiRand version": "1K for development, 27K for full experiment", "Possible metrics": "Recall@K, NDCG@K, HitRate@K, MRR"},
    {"Task": "top-K recommendation", "Target": "held-out positive item", "Inputs": "user history, item universe, candidate set", "Recommended KuaiRand version": "1K then 27K", "Possible metrics": "Recall@K, NDCG@K, HitRate@K"},
    {"Task": "click prediction", "Target": "is_click", "Inputs": "user, item, context, history", "Recommended KuaiRand version": "Pure/1K for EDA, 27K for scale", "Possible metrics": "AUC, LogLoss"},
    {"Task": "long-view prediction", "Target": "long_view", "Inputs": "watch-time context and item metadata", "Recommended KuaiRand version": "Pure/1K", "Possible metrics": "AUC, LogLoss, calibration"},
    {"Task": "multi-task feedback prediction", "Target": "click, like, follow, comment, forward, hate, long_view", "Inputs": "shared user/item/context representation", "Recommended KuaiRand version": "1K then 27K", "Possible metrics": "AUC/LogLoss per target, ranking metrics for derived preference"},
    {"Task": "debiased recommendation", "Target": "policy-aware positive feedback", "Inputs": "standard logs plus random exposure", "Recommended KuaiRand version": "Pure and 27K", "Possible metrics": "IPS/SNIPS variants, NDCG@K under defined candidate policy"},
    {"Task": "off-policy evaluation", "Target": "counterfactual policy value", "Inputs": "randomized exposure and logged feedback", "Recommended KuaiRand version": "Pure/27K", "Possible metrics": "IPS, SNIPS, DR if propensities/design allow"},
    {"Task": "random-exposure evaluation", "Target": "feedback under random candidate exposure", "Inputs": "random log", "Recommended KuaiRand version": "Pure/27K", "Possible metrics": "AUC, Recall@K/NDCG@K with explicit candidate set"},
    {"Task": "causal recommendation", "Target": "causal estimand defined before modeling", "Inputs": "random exposure, standard exposure, covariates", "Recommended KuaiRand version": "Pure/27K", "Possible metrics": "Estimand-specific error/value metrics"},
])
display(tasks)


## Metrics

Potential metrics include:

- `Recall@K`
- `NDCG@K`
- `HitRate@K`
- `MRR`
- `AUC` / `LogLoss` where prediction probabilities are the objective

The same metric name does not imply a comparable benchmark. For example, `NDCG@10` depends on split, candidate set, negative sampling, filtering, evaluation users, preprocessing, repeated-item policy, and whether random exposure is part of evaluation. This is critical for any future SOTA table.


# What We Learned About KuaiRand

The cell below builds conclusions from computed tables when available. If the notebook is run without KuaiRand files, it reports that computed conclusions are pending instead of inventing numbers.


In [ ]:
conclusions = [
    "Documentation / README statistics show that 1K is the practical development subset, 27K is the full-scale experiment candidate, and Pure focuses on the common/random candidate-pool setting.",
    "KuaiRand provides both standard-policy logs and randomized exposure logs; these should be modeled as different collection policies, not renamed automatically to train/test.",
    "Random exposure is important for studying exposure bias, selection bias, debiasing, off-policy evaluation, and causal recommendation, subject to a task-specific design.",
    "The notebook treats video statistic features as potentially leaky aggregated features until their time window is verified.",
    "Benchmark protocol choice is central: preprocessing, split, candidates, negative sampling, and evaluation users must match the selected published baseline family.",
]

if not policy_counts_df.empty:
    for version in sorted(policy_counts_df["Version"].unique()):
        sub = policy_counts_df[policy_counts_df["Version"] == version]
        policies = ", ".join(sorted(sub["Policy"].astype(str).tolist()))
        conclusions.append(f"Computed from local files: KuaiRand-{version} contains policy summaries for: {policies}.")
else:
    conclusions.append("Computed policy counts are pending because Pure/1K files were not available in DATA_ROOT.")

if not sequence_summary_df.empty:
    row = sequence_summary_df.iloc[0]
    conclusions.append(
        "Computed from local files: 1K standard user-history length summary is available "
        f"with median={row.get('median')} and p95={row.get('p95')} interactions."
    )
else:
    conclusions.append("Computed sequence-length conclusions are pending until KuaiRand-1K is available.")

if not sparsity_summary_df.empty:
    sparsity_row = sparsity_summary_df[sparsity_summary_df["quantity"].str.contains("sparsity", na=False)]
    if not sparsity_row.empty:
        conclusions.append(
            "Computed from local files: approximate interaction-matrix sparsity over the observed 1K standard universe is "
            f"{sparsity_row.iloc[0]['mean']}."
        )
else:
    conclusions.append("Computed sparsity and long-tail conclusions are pending until KuaiRand-1K standard logs are available.")

if not feedback_rates_df.empty:
    available_signals = ", ".join(sorted(feedback_rates_df["signal"].unique()))
    conclusions.append(f"Computed from local files: feedback summary is available for {available_signals}.")
else:
    conclusions.append("Computed feedback-rate conclusions are pending until interaction logs are available.")

if not shift_summary_df.empty:
    conclusions.append("Computed from local files: standard-vs-random distribution-difference summary is available for inspected versions.")
else:
    conclusions.append("Standard-vs-random distribution-difference conclusions are pending until Pure/1K logs are available.")

conclusions.extend([
    "Pure is useful for candidate-pool and random-exposure-focused studies.",
    "1K is useful for interactive development, protocol debugging, and baseline reproduction checks.",
    "27K should be used for full-scale experiments through batch/lazy scripts rather than full interactive notebook scans.",
    "Before baseline work, choose concrete published papers and reproduce their preprocessing, split, candidate construction, and evaluation exactly.",
])

conclusions = conclusions[:15]
display(Markdown("\n".join(f"{idx + 1}. {text}" for idx, text in enumerate(conclusions))))


# Next Steps

1. Choose a published benchmark family.
2. Choose a concrete strong baseline / SOTA paper.
3. Reproduce preprocessing.
4. Reproduce the split.
5. Reproduce the evaluation protocol.
6. Obtain baseline metrics.
7. Compare with published results.
8. Only then design a new method.
9. Run ablations.
10. Move to full 27K experiments.


## 27K Execution Note

The full 27K scan should be run with `src/eda_27k.py`, preferably through `slurm/eda_27k.sh` after cluster-specific resources are filled in. The notebook avoids `pandas.read_csv` on 27K and does not materialize the full 27K interaction logs.
